# Do Tumors Recapitulate Fetal Mitotic Gene Expression Programs?

This notebook investigates whether tumor gene expression profiles resemble fetal developmental programs, focusing on mitotic genes.

We compare tumor and fetal RNA-seq data using:
- PCA for global structure
- correlation-based distance for pattern similarity

The goal is to determine whether tumor proliferation reflects fetal-like transcriptional states.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

tpm_path = "../data/tcga/TcgaTargetGtex_rsem_gene_tpm.gz"
pheno_path = "../data/tcga/TcgaTargetGTEX_phenotype.txt"
mitosis_path = "../data/mitosis/mitosis_ensembl_ids.txt"
fetal_path = "../data/fetal/fetal_mitotic_tpm.tsv"

## Load Metadata and Define the Mitotic Gene Set

In [2]:
pheno = pd.read_csv(pheno_path, sep="\t", encoding="latin1")
tumor_samples = pheno[pheno["_sample_type"] == "Primary Tumor"]["sample"].tolist()

mitotic_genes = pd.read_csv(mitosis_path, header=None)[0]
mitotic_genes = mitotic_genes.astype(str).str.split(".").str[0]

print(f"Number of tumor samples: {len(tumor_samples)}")
print(f"Mitotic genes: {len(mitotic_genes)}")

Number of tumor samples: 9185
Mitotic genes: 662


## Construct the Tumor Mitotic Expression Matrix

The TCGA matrix is processed in chunks to retain only primary tumor samples and mitotic genes.

In [ ]:
chunks = []
chunk_size = 2000

for i, chunk in enumerate(
    pd.read_csv(
        tpm_path,
        sep="\t",
        compression="gzip",
        chunksize=chunk_size,
        index_col=0
    )
):
    chunk.index = chunk.index.astype(str).str.split(".").str[0]
    chunk = chunk[chunk.index.isin(mitotic_genes)]

    if not chunk.empty:
        keep_cols = [c for c in chunk.columns if c in tumor_samples]
        chunk = chunk[keep_cols]
        chunks.append(chunk)

    if i % 10 == 0:
        print(f"Processed chunk {i+1}")

tumor_df = pd.concat(chunks)
tumor_df = tumor_df.groupby(tumor_df.index).mean()

print("Tumor matrix shape:", tumor_df.shape)

tumor_df.to_csv("../data/tcga/primary_tumor_only_mitotic_tpm.csv")
print("Saved tumor mitotic matrix.")

Processed chunk 1


## Load and Clean the Fetal Mitotic Expression Matrix

In [ ]:
fetal_df = pd.read_csv(fetal_path, sep="\t", index_col=0)

fetal_df.index = fetal_df.index.astype(str).str.split(".").str[0]
fetal_df = fetal_df.drop(columns=["Gene Name"])
fetal_df = fetal_df.groupby(fetal_df.index).mean()

print("Fetal matrix shape:", fetal_df.shape)

## Align Tumor and Fetal Matrices on Shared Genes

In [ ]:
shared_genes = fetal_df.index.intersection(tumor_df.index)

fetal_aligned = fetal_df.loc[shared_genes].copy()
tumor_aligned = tumor_df.loc[shared_genes].copy()

tumor_aligned = tumor_aligned.apply(pd.to_numeric, errors="coerce").fillna(0)
fetal_aligned = fetal_aligned.apply(pd.to_numeric, errors="coerce").fillna(0)

print("Shared genes:", len(shared_genes))
print("Aligned fetal shape:", fetal_aligned.shape)
print("Aligned tumor shape:", tumor_aligned.shape)

## Balanced PCA of Tumor and Fetal Samples

Because tumor samples greatly outnumber fetal samples, tumors are downsampled to match fetal sample size before PCA.

In [ ]:
fetal_T = fetal_aligned.T.copy()
tumor_T = tumor_aligned.T.copy()

fetal_T["source"] = "Fetal"
tumor_T["source"] = "Tumor"

tumor_sampled = tumor_T.sample(n=len(fetal_T), random_state=42)

combined_df = pd.concat([fetal_T, tumor_sampled], axis=0)
labels = combined_df["source"].copy()
combined_df = combined_df.drop(columns=["source"])

X_scaled = StandardScaler().fit_transform(combined_df)

pca = PCA(n_components=2)
pcs = pca.fit_transform(X_scaled)

joint_pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2"])
joint_pca_df["source"] = labels.values

print("Explained variance:", pca.explained_variance_ratio_)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=joint_pca_df,
    x="PC1",
    y="PC2",
    hue="source",
    alpha=0.7,
    s=40
)

plt.title("Tumor vs Fetal Mitotic Gene Expression (Balanced PCA)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.legend(title="Sample Type")
plt.tight_layout()
plt.savefig("../results/figures/joint_pca_balanced.png", dpi=300, bbox_inches="tight")
plt.show()

## Interpretation

After balancing sample sizes, tumor samples cluster tightly within a narrow region of mitotic gene expression space, while fetal samples span a much broader range.

This suggests that tumor proliferation reflects a constrained subset of developmental transcriptional states rather than broadly recapitulating fetal programs.

## Build Fetal Stage–Tissue Reference Profiles

In [ ]:
fetal_meta = pd.DataFrame({"sample": fetal_aligned.columns})

def parse_fetal_label(label):
    parts = str(label).split(",", 1)
    stage = parts[0].strip() if len(parts) > 0 else "Unknown"
    tissue = parts[1].strip() if len(parts) > 1 else "Unknown"
    return pd.Series({"stage": stage, "tissue": tissue})

fetal_meta[["stage", "tissue"]] = fetal_meta["sample"].apply(parse_fetal_label)
fetal_meta["stage_tissue"] = fetal_meta["stage"] + " | " + fetal_meta["tissue"]

fetal_samples_by_gene = fetal_aligned.T.copy()
fetal_samples_by_gene["stage_tissue"] = fetal_meta["stage_tissue"].values

fetal_centroids = fetal_samples_by_gene.groupby("stage_tissue").mean().T

print("Fetal centroids shape:", fetal_centroids.shape)

## Tumor-to-Fetal Similarity Using Correlation Distance

Tumor samples are compared to fetal stage–tissue centroids using correlation distance, which captures similarity in expression pattern rather than absolute magnitude.

In [ ]:
tumor_subset = tumor_aligned.sample(n=500, axis=1, random_state=42)

shared_for_dist = tumor_subset.index.intersection(fetal_centroids.index)

tumor_for_dist = tumor_subset.loc[shared_for_dist].T
fetal_for_dist = fetal_centroids.loc[shared_for_dist].T

tumor_scaled = StandardScaler().fit_transform(tumor_for_dist)
fetal_scaled = StandardScaler().fit_transform(fetal_for_dist)

distance_matrix = cdist(
    tumor_scaled,
    fetal_scaled,
    metric="correlation"
)

distance_df = pd.DataFrame(
    distance_matrix,
    index=tumor_for_dist.index,
    columns=fetal_for_dist.index
)

print("Distance matrix shape:", distance_df.shape)

In [ ]:
closest_match = distance_df.idxmin(axis=1).rename("closest_fetal_profile")
closest_distance = distance_df.min(axis=1).rename("distance")

tumor_match_df = pd.concat([closest_match, closest_distance], axis=1).reset_index()
tumor_match_df = tumor_match_df.rename(columns={"index": "tumor_sample"})

tumor_match_df["stage"] = tumor_match_df["closest_fetal_profile"].str.split(" \\| ").str[0]
tumor_match_df["tissue"] = tumor_match_df["closest_fetal_profile"].str.split(" \\| ").str[1]

print(tumor_match_df["tissue"].value_counts().head(10))

tumor_match_df.to_csv("../results/tables/tumor_fetal_matches_corr.csv", index=False)
print("Saved correlation-based matching results.")

In [ ]:
top_tissues = tumor_match_df["tissue"].value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_tissues.values, y=top_tissues.index)

plt.title("Tumor → Fetal Tissue Similarity (Correlation Distance)")
plt.xlabel("Number of Tumors with Nearest Match")
plt.ylabel("Fetal Tissue")

plt.tight_layout()
plt.savefig("../results/figures/top_fetal_matches_corr.png", dpi=300, bbox_inches="tight")
plt.show()

## Interpretation

Tumors map most frequently to fetal tissues associated with strong proliferative or developmentally active contexts, including kidney, testis, liver, and neural tissues.

Rather than matching a single fetal state, tumors distribute across multiple developmental contexts, suggesting that cancer proliferation reflects a restricted and heterogeneous subset of developmental programs.